In [1]:
"Hello World"

'Hello World'

In [2]:
GPT_CONFIG_124M={
    "vocab_size":50257,
    "context_length":1024,
    "emb_dim":768,
    "n_heads":12,
    "n_layers":12,
    "drop_rate":0.1,
    "qkv_bias":False
}

In [3]:
import torch
import torch.nn as nn

In [12]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

In [4]:
class LinearNorm(nn.Module):
  def __init__(self,emb_dim):
    super().__init__()
    self.eps=1e-5
    self.scale=nn.Parameter(torch.ones(emb_dim))
    self.shift=nn.Parameter(torch.zeros(emb_dim))


  def forward(self,x):
      mean=x.mean(dim=-1,keepdim=True)
      var=x.var(dim=-1,keepdim=True,unbiased=False)
      norm_x=(x-mean)/torch.sqrt(var+self.eps)
      return self.scale*norm_x + self.shift


In [10]:
class GELU(nn.Module):
  def __init__(self):
    super().__init__()
    self.gelu=nn.GELU()

  def forward(self,x):
    # 0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2/torch.pi))*(x+0.044715*torch.pow(x,3))))
    return self.gelu(x)

In [11]:
class FeedForward(nn.Module):
    def __init__(self,emb_dim):
        super().__init__()
        self.layers=nn.Sequential(
            nn.Linear(emb_dim,4*emb_dim),
            GELU(),
            nn.Linear(4*emb_dim,emb_dim)
        )

    def forward(self,x):
        return self.layers(x)


In [14]:
class TransformerBlock(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.linear_norm1=LinearNorm(config["emb_dim"])
    self.linear_norm2=LinearNorm(config["emb_dim"])
    self.attention=MultiHeadAttention(
        d_in=config["emb_dim"],
        d_out=config["emb_dim"],
        context_length=config["context_length"],
        dropout=config["drop_rate"],
        num_heads=config["n_heads"],
        qkv_bias=config["qkv_bias"]
    )

    self.feed_forward=FeedForward(config["emb_dim"])
    self.drop_shortcut=nn.Dropout(config['drop_rate'])

  def forward(self,x):
    x=x+self.drop_shortcut(self.attention(self.linear_norm1(x)))
    x=x+self.drop_shortcut(self.feed_forward(self.linear_norm2(x)))
    return x

In [15]:
torch.manual_seed(42)
x=torch.rand(2,4,768)
block=TransformerBlock(GPT_CONFIG_124M)

output=block(x)

print("Input shape: ",x.shape)
print("Output shape: ",output.shape)

Input shape:  torch.Size([2, 4, 768])
Output shape:  torch.Size([2, 4, 768])


In [49]:
class GPTModel(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.token_em=nn.Embedding(config["vocab_size"],config["emb_dim"])
    self.pos_em=nn.Embedding(config["context_length"],config["emb_dim"])
    self.blocks=nn.Sequential(*[TransformerBlock(config) for _ in range(config["n_layers"])])

    self.final_linear_norm=LinearNorm(config["emb_dim"])
    self.output_head=nn.Linear(config["emb_dim"],config["vocab_size"],bias=False)

    # self.token_em.weight=self.output_head.weight # complexcity reduces
  def forward(self,x):
    batch_size,seq_len=x.shape
    x=self.token_em(x)
    x=x+self.pos_em(torch.arange(seq_len,device=x.device))

    x=self.blocks(x)

    x=self.final_linear_norm(x)
    logits=self.output_head(x)

    return logits


In [50]:
model=GPTModel(config=GPT_CONFIG_124M)

In [51]:
torch.manual_seed(42)
batch = torch.randint(0, GPT_CONFIG_124M['vocab_size'], (2, 4), dtype=torch.long)

out = model(batch)

print("input batch", batch.shape)
print("Output batch", out.shape)

input batch torch.Size([2, 4])
Output batch torch.Size([2, 4, 50257])


In [52]:
model.token_em.weight.shape

torch.Size([50257, 768])

In [53]:
model.output_head.weight.shape

torch.Size([50257, 768])

In [54]:
!pip install torchinfo
from torchinfo import summary

In [55]:
summary(model,input_size=(2,4), dtypes=[torch.long])

Layer (type:depth-idx)                        Output Shape              Param #
GPTModel                                      [2, 4, 50257]             --
├─Embedding: 1-1                              [2, 4, 768]               38,597,376
├─Embedding: 1-2                              [4, 768]                  786,432
├─Sequential: 1-3                             [2, 4, 768]               --
│    └─TransformerBlock: 2-1                  [2, 4, 768]               --
│    │    └─LinearNorm: 3-1                   [2, 4, 768]               1,536
│    │    └─MultiHeadAttention: 3-2           [2, 4, 768]               2,360,064
│    │    └─Dropout: 3-3                      [2, 4, 768]               --
│    │    └─LinearNorm: 3-4                   [2, 4, 768]               1,536
│    │    └─FeedForward: 3-5                  [2, 4, 768]               4,722,432
│    │    └─Dropout: 3-6                      [2, 4, 768]               --
│    └─TransformerBlock: 2-2                  [2, 4, 768]     

In [65]:
def generate_text_simple(
    model,
    idx,
    max_new_tokens=100,
    context_size=GPT_CONFIG_124M['context_length']
):

  for _ in range(max_new_tokens):
      # crop idx to the last context_size tokens
      idx_cond = idx[:, -context_size:] # Fixed: Changed to slice notation

      with torch.no_grad():
          logits = model(idx_cond)

      last_batch_next=logits[:,-1,:]

      last_batch_next=torch.softmax(last_batch_next,dim=-1)

      idx_next=torch.argmax(last_batch_next, dim=-1).unsqueeze(-1) # Fixed: ensure idx_next is 2D for concatenation

      idx=torch.cat((idx,idx_next),dim=1)

  return idx

In [57]:
import tiktoken

enc=tiktoken.get_encoding("gpt2")

In [61]:
start_context="Hello, I am"
encoded=enc.encode(start_context)
print(encoded)
encoded_tensor=torch.tensor(encoded,dtype=torch.long).unsqueeze(0)
print(encoded_tensor)

[15496, 11, 314, 716]
tensor([[15496,    11,   314,   716]])


In [64]:
model.eval()
out=generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M['context_length']
)
print(
    "Output",out,"shape",out.shape
)

Output tensor([[15496,    11,   314,   716,  1778, 17961, 10753,  9708, 34563,  2337]]) shape torch.Size([1, 10])


In [67]:
decoded_text=enc.decode(out.squeeze().tolist())
print(decoded_text)

Hello, I am SuINAL oversphonesIBLEights
